# Resultado final

Guarda el resultado de la unión de las tablas Persona, Hogar y Vivienda en una sola tabla denormalizada

## Inicializa Spark

In [0]:
%run "/Workspace/Project/etl_censo_databricks/notebooks/00_Init_Spark"

## Lee tablas con datos

In [0]:
df_vivienda = spark.read.table("silver.viviendas")

In [0]:
df_hogar = spark.read.table("silver.hogares")

In [0]:
df_persona = spark.read.table("silver.personas")

## Unión entre las tablas

In [0]:
# Obtengo las columnas que se repiten entre hogar y persona, y vivienda y persona
df_vivienda_set = set(df_vivienda.columns)
df_hogar_set = set(df_hogar.columns)
df_persona_set = set(df_persona.columns)

df_pv_inter = df_persona_set.intersection(df_vivienda_set)
df_ph_inter = df_persona_set.intersection(df_hogar_set)

df_pv = [df_vivienda[c] for c in list(df_pv_inter)]
df_ph = [df_hogar[c] for c in list(df_ph_inter)]

In [0]:
df_persona_uni = df_persona \
    .join(df_vivienda, ["id_vivienda"], "left") \
    .join(df_hogar, ["id_vivienda", "id_hogar"], "left")

In [0]:
# Elimina los campos duplicados
df_persona_uni = df_persona_uni.drop(*df_pv)
df_persona_uni = df_persona_uni.drop(*df_ph)
display(df_persona_uni)

## Guarda resultado en la tabla final

In [0]:
df_persona_uni.write.insertInto("gold.personas", overwrite=True)